# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

### Imports

In [ ]:
from src.utils.video import open_video, get_frames
from src.utils.visualization import show_images, show_detection_table, show_identity_table, show_hota_table
from src.utils.model_loading import load_fine_tuned_yolo_model
from src.utils.annotations.download_annotations import download_annotations
from src.utils.annotations.load_annotations import load_annotations

from src.types.tracking import merge_detections

from src.detection.mog2_detection import run_mog2_detection, mog2_to_detection_output
from src.detection.yolo_detection import run_yolo_detection, yolo_to_detection_output
from src.tracking.sort import apply_sort
from src.tracking.deep_sort import apply_deep_sort
from src.detection.nms import class_independent_nms
from src.tracking.label_resolution import resolve_track_labels

from src.evaluation.evaluate_tracking import evaluate_tracking
from src.evaluation.evaluate_detection import evaluate_detection

from collections import defaultdict
import cv2
import time
import os
from pathlib import Path
from dotenv import load_dotenv
from ultralytics import YOLO

### Global Parameters

In [ ]:
INSPECT_CAMERA_ID = "cam_13"                                # The camera we will inspect in detail in the notebook
FRAME_STRIDE = 5                                            # Step between GT annotation frames and prediction frame indices
MAX_FRAMES   = None                                         # Frames to load per camera (None = all frames)

VIDEOS_DIR = "data/videos"                                  # Directory where the input videos are stored
CAMERA_SETTINGS_DIR = "data/camera_settings"                # Directory where the camera settings JSON files are stored
MODELS_DIR = "models/"                                      # Directory where pretrained and fine-tuned models are stored
FINE_TUNED_MODELS_DIR = f"{MODELS_DIR}/fine_tuned_models"   # Directory where fine-tuned models will be saved
ANNOTATIONS = "data/annotations/tracking_01"                # Directory where the tracking annotations are stored (after downloading and extraction)

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

### Open Video and Read Frames

In [ ]:
frames_color = {}
fps = {}
for cam_id, cam in CAMERAS.items():
    # Load the video 
    cap = open_video(cam["video_path"])

    # Extract the frames
    frames_color[cam_id], _ = get_frames(cap, max_frames=MAX_FRAMES)  

    # Get the frames per second (fps) of the video
    fps[cam_id] = cap.get(cv2.CAP_PROP_FPS)

    # Release the video capture object to free resources
    cap.release()
    print(f"[{cam_id}] Loaded {len(frames_color[cam_id])} frames at {fps[cam_id]:.1f} fps")

show_images(frames_color[INSPECT_CAMERA_ID])

---

# Detection 

---

## Background Subtraction with MOG2 for Player Detection

> Manafifard, M., Ebadi, H., & Abrishami Moghaddam, H. (2017). A survey on player tracking in soccer videos. *Computer Vision and Image Understanding*

We use the MOG2 (Mixture of Gaussians v2) background subtractor to detect moving players in each frame.
As a background subtraction method, MOG2 resulted as a good choice for our application and through various experiments with other methods.

We further improved the detection results by applying pre-processing steps and post-processing steps to the frames and the foreground masks, respectively.


For **pre-processing**, we applied a illumination normalization technique called **CLAHE (Contrast Limited Adaptive Histogram Equalization)** to enhance the contrast of the frames, which helps in better foreground detection. Then to limit the noise generated from the shadow generated from the high reflective surfaces in the field, we removed from the detection mask the pixels that correspond to the hue values of the field.

For **post-processing**, we applied morphological operations such as opening and closing to remove noise and fill holes in the detected foreground masks, and then we applied contour detection to extract the bounding boxes of the detected players. We removed small contours that are likely to be noise by filtering based on contour area.

### Parameters




In [ ]:
MOG2_LEARNING_RATE  = -1        # -1 lets OpenCV adapt the learning rate automatically
MOG2_HISTORY        = 1000      # number of frames used to model the background
MOG2_VAR_THRESHOLD  = 25        # Mahalanobis distance threshold for foreground/background decision
MOG2_OPENING_KERNEL = 7         # morphological opening kernel size (noise removal)
MOG2_CLOSING_KERNEL = 11        # morphological closing kernel size (hole filling)
MOG2_MIN_AREA       = 1500      # minimum contour area to keep (px²)
MOG2_MAX_AREA       = 200_000   # maximum contour area to keep (px²)

### Run MOG2 Detection on All Frames

In [ ]:
mog2_masks = {}
mog2_detection_output = {}

for cam_id in CAMERAS:
    
    # Run MOG2 detection 
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with MOG2 detection...")
    start_time = time.time()
    mog2_masks[cam_id] = run_mog2_detection(
        frames_color[cam_id],
        learning_rate=MOG2_LEARNING_RATE,
        history_length=MOG2_HISTORY,
        var_threshold=MOG2_VAR_THRESHOLD,
        opening_kernel_size=MOG2_OPENING_KERNEL,
        closing_kernel_size=MOG2_CLOSING_KERNEL,
        min_area=MOG2_MIN_AREA,
        max_area=MOG2_MAX_AREA,
    )
    elapsed = time.time() - start_time

    # Convert MOG2 masks to DetectionOutput format for evaluation and tracking
    mog2_detection_output[cam_id] = mog2_to_detection_output(mog2_masks[cam_id], camera_id=cam_id, fps=fps[cam_id])
    n_dets = sum(f.num_detections for f in mog2_detection_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {n_dets} detections across {len(mog2_detection_output[cam_id].frames)} frames")

# Show raw masks for the inspection camera (skip warm-up frames)
show_images(mog2_masks[INSPECT_CAMERA_ID][15:])

### Inspect MOG2 Detections

We visualize the MOG2 blob detected by outputting the binary mask.
This allows us to inspect the quality of the foreground detection and to tune the parameters of the MOG2 background subtractor and the post-processing steps accordingly.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], mog2_detection_output[INSPECT_CAMERA_ID])

---

## Pre-Trained YOLOv11

> Jocher, G., Qiu, J., & Chaurasia, A. (2023). Ultralytics YOLO. 
>
> Joseph Redmon and Santosh Divvala and Ross Girshick and Ali Farhadi. (2016). You Only Look Once: Unified, Real-Time Object Detection.

We use a deep learning-based object detection model, YOLOv11, to detect players in each frame. This can be done by loading a pre-trained YOLOv11 model and running inference on the frames to get bounding boxes for detected players.

### Parameters

In [ ]:
YOLO_PRETRAINED_CONF = 0.4    # confidence threshold
YOLO_PRETRAINED_IOU  = 0.45   # IoU threshold for built-in NMS
YOLO_PRETRAINED_SIZE = 640    # inference image size

### Models

In [ ]:
# Resolve the path of pretrained yolo model
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
pretrained_yolo_model_path = os.path.join(MODELS_DIR, "yolo11m.pt")

# Load the YOLOv11m model
pretrained_yolo_model = YOLO(pretrained_yolo_model_path)    

### Run YOLO Detection on All Frames

Use YOLOv11's built-in `.predict()` method to detect objects in each frame. The output will include bounding boxes, confidence scores, and class labels for each detected object.

**Output per frame:**
- Bounding boxes: `[x1, y1, x2, y2]` (pixel coordinates)
- Confidence scores: Detection confidence (0–1)
- Class labels: What was detected (e.g., "person", "sports_ball")

In [ ]:
yolo_pretrained_raw = {}

for cam_id in CAMERAS:
    # Run YOLOv11m detection
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with pre-trained YOLO detection...")
    start_time = time.time()
    yolo_pretrained_raw[cam_id] = run_yolo_detection(
        pretrained_yolo_model, frames_color[cam_id],
        conf_threshold=YOLO_PRETRAINED_CONF,
        iou_threshold=YOLO_PRETRAINED_IOU,
        inference_size=YOLO_PRETRAINED_SIZE,
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_pretrained_raw[cam_id])} raw detections")

In [ ]:
yolo_pretrained_detection_output = {}
for cam_id in CAMERAS:
    # Convert YOLO raw detections to DetectionOutput format for evaluation and tracking
    yolo_pretrained_detection_output[cam_id] = yolo_to_detection_output(
        yolo_pretrained_raw[cam_id], pretrained_yolo_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_pt",
    )

    # Count total detections after conversion
    n_dets = sum(f.num_detections for f in yolo_pretrained_detection_output[cam_id].frames)
    print(f"[{cam_id}] Total detections after conversion: {n_dets} across {len(yolo_pretrained_detection_output[cam_id].frames)} frames")

### Inspect YOLO Detections

In [ ]:
cam = INSPECT_CAMERA_ID
print(f"First frame detections [{cam}]: {yolo_pretrained_detection_output[cam].frames[0].num_detections} objects detected")
for i, det in enumerate(yolo_pretrained_detection_output[cam].frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={det.class_name}, Confidence={det.confidence:.2f}, BBox={det.bbox}")

show_images(frames_color[cam], yolo_pretrained_detection_output[cam])

---

## Fine-Tuned YOLOv11

> Jocher, G., Qiu, J., & Chaurasia, A. (2023). Ultralytics YOLO. [`yolo`]
>
> J. Redmon and S. Divvala and R. Girshick and Ali Farhadi. (2016). You Only Look Once: Unified, Real-Time Object Detection.

We fine-tuned a YOLOv11m model on our annotated dataset to detect players with identity labels (e.g. `Red_11`, `White_7`) and the ball as separate classes. A two-pass inference strategy is used: a standard-resolution pass for players and a high-resolution pass for the ball.

Let's check the labels in the model to ensure they match our dataset's classes. This is crucial for accurate training and inference.

### Parameters

In [ ]:
YOLO_FINETUNED_PLAYER_CONF = 0.3    # confidence threshold for player pass
YOLO_FINETUNED_BALL_CONF   = 0.1    # lower threshold for ball (small object, favour recall)
YOLO_FINETUNED_PLAYER_SIZE = 640    # inference size for player pass
YOLO_FINETUNED_BALL_SIZE   = 1280   # higher resolution for ball pass

### Models

In [ ]:
# Script to load the fine-tuned YOLO model for player pass detection
yolo_finetuned_model = load_fine_tuned_yolo_model()

### Run Fine-Tuned YOLO Detection on All Frames

In [ ]:
yolo_finetuned_player_raw = {}
for cam_id in CAMERAS:
    # Run fine-tuned YOLO detection for player pass
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with fine-tuned YOLO (player pass)...")
    start_time = time.time()
    yolo_finetuned_player_raw[cam_id] = run_yolo_detection(
        yolo_finetuned_model,
        frames_color[cam_id],
        conf_threshold=YOLO_FINETUNED_PLAYER_CONF,
        inference_size=YOLO_FINETUNED_PLAYER_SIZE,
        class_ids=list(range(1, len(yolo_finetuned_model.names))),
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_finetuned_player_raw[cam_id])} player detections")

### Higher-Resolution Pass — Ball Only

The football is a small object (~10–20 px in match footage) and the standard 640-px inference grid struggles to localize it. We therefore run a **second** inference pass at higher `imgsz` (1280) with a lower confidence threshold.

Critically, this pass is **class-filtered to the ball only**. Running high-resolution inference for `person` would also surface tiny bench players and spectators in the stands — exactly what we don't want.

In [ ]:
yolo_finetuned_ball_raw = {}
for cam_id in CAMERAS:
    # Run fine-tuned YOLO detection for ball pass
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with fine-tuned YOLO (ball pass)...")
    start_time = time.time()
    yolo_finetuned_ball_raw[cam_id] = run_yolo_detection(
        yolo_finetuned_model, frames_color[cam_id],
        conf_threshold=YOLO_FINETUNED_BALL_CONF,
        inference_size=YOLO_FINETUNED_BALL_SIZE,
        class_ids=[0],
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_finetuned_ball_raw[cam_id])} ball detections")

### Merge Player and Ball Detections

Combine both passes into a single `DetectionOutput`, frame by frame. After fine-tuning, only the `class_ids` lists in the two passes above need to change — the merge stays identical.

In [ ]:
yolo_finetuned_merged_output = {}
for cam_id in CAMERAS:
    # Convert player and ball output
    player_out = yolo_to_detection_output(
        yolo_finetuned_player_raw[cam_id], yolo_finetuned_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_ft",
    )
    ball_out = yolo_to_detection_output(
        yolo_finetuned_ball_raw[cam_id], yolo_finetuned_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_ft",
    )

    # Merge both converted outputs
    yolo_finetuned_merged_output[cam_id] = merge_detections(player_out, ball_out)

    # Count total detections after merging
    n_merged = sum(f.num_detections for f in yolo_finetuned_merged_output[cam_id].frames)
    print(f"[{cam_id}] Merged: {n_merged} detections across {len(yolo_finetuned_merged_output[cam_id].frames)} frames")

### Inspect Merged Detections

In [ ]:
cam = INSPECT_CAMERA_ID
print(f"First frame detections [{cam}]: {yolo_finetuned_merged_output[cam].frames[0].num_detections} objects detected")
for i, det in enumerate(yolo_finetuned_merged_output[cam].frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={det.class_name}, Confidence={det.confidence:.2f}, BBox={det.bbox}")

show_images(frames_color[cam], yolo_finetuned_merged_output[cam])

---

## Class-Agnostic NMS

> Neubeck, A., & Van Gool, L. (2006). Efficient non-maximum suppression. *ICPR'06*, Vol. 3, 850–855. [`nms`]

YOLO does NMS only within each class. Our identity-encoded labels (`Red_5`, `Red_11`, …) make this insufficient: two boxes for the same physical player but different predicted identities both survive, then spawn parallel tracks in DeepSORT. We collapse them with a single class-agnostic greedy NMS pass before tracking — for each frame, sort detections by confidence and keep a box only if it doesn't overlap an already-kept box above the IoU threshold.

### Parameters

In [ ]:
NMS_IOU_THRESHOLD = 0.5    # IoU above which a lower-confidence box is suppressed

### Run YOLO Detection with Class-Agnostic NMS on All Frames

In [ ]:
yolo_finetuned_detection_output = {}
for cam_id in CAMERAS:
    # Apply class-agnostic NMS to the merged detections
    before = sum(len(f.detections) for f in yolo_finetuned_merged_output[cam_id].frames)
    yolo_finetuned_detection_output[cam_id] = class_independent_nms(yolo_finetuned_merged_output[cam_id], iou_threshold=NMS_IOU_THRESHOLD)
    after = sum(len(f.detections) for f in yolo_finetuned_detection_output[cam_id].frames)
    print(f"[{cam_id}] Class-agnostic NMS: {before} → {after} detections ({before - after} suppressed)")

### Inspect YOLO Detections

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], yolo_finetuned_detection_output[INSPECT_CAMERA_ID])

---

# Tracking

---

## SORT Tracking

> Bewley, A., Ge, Z., Ott, L., Ramos, F., & Upcroft, B. (2016). Simple online and realtime tracking. *ICIP 2016*, 3464–3468. [`sort`]

SORT (Simple Online and Realtime Tracking) assigns a stable `track_id` to each detection using a **Kalman filter** for motion prediction and the **Hungarian algorithm** for IoU-based assignment. It operates with no appearance features — only bounding box overlap.

### Parameters

In [ ]:
SORT_MAX_IOU_DISTANCE = 0.8   # IoU gating threshold for Kalman-prediction-to-detection assignment
SORT_MAX_AGE          = 60    # frames a track stays alive while unmatched (~2.4 s at 25 fps)
SORT_N_INIT           = 2     # consecutive matches required before a track is confirmed

### Run SORT Tracking on All Frames

In [ ]:
sort_tracking_output = {}
for cam_id in CAMERAS:
    # Apply SORT tracking to the fine-tuned YOLO detections
    print(f"[{cam_id}] Running SORT on {len(yolo_finetuned_detection_output[cam_id].frames)} frames...")
    start_time = time.time()
    sort_tracking_output[cam_id] = apply_sort(
        yolo_finetuned_detection_output[cam_id],
        max_iou_distance=SORT_MAX_IOU_DISTANCE,
        max_age=SORT_MAX_AGE,
        n_init=SORT_N_INIT,
    )
    elapsed = time.time() - start_time

    # Count total tracks and detections after tracking
    total_tracks = len({d.track_id for f in sort_tracking_output[cam_id].frames for d in f.detections})
    total_dets   = sum(len(f.detections) for f in sort_tracking_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {total_tracks} tracks across {total_dets} tracked detections")

### Inspect SORT Tracks

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], sort_tracking_output[INSPECT_CAMERA_ID])

### Resolve Per-Track Labels

YOLO's class names already encode identity (e.g. `Red_11` = Red team #11), but the model occasionally flips a player's label between frames or returns two boxes with the same class in one frame. We use the track ids assigned above to vote for a stable label per track and resolve collisions where two tracks claim the same identity.

The track with the highest cumulative confidence keeps its top-choice label; any other track that wanted that label falls through to its next-best class. `Ball` is exempt — it's a category, not a unique identity.

In [ ]:
sort_resolved_output = {}
for cam_id in CAMERAS:
    # Resolve track labels by majority voting on the detections associated with each track
    sort_resolved_output[cam_id] = resolve_track_labels(sort_tracking_output[cam_id])

    # Analyze label stability per track (expecting most tracks to have a single stable label, ideally "Ball" or "Player")
    per_track_label = defaultdict(set)
    for fd in sort_resolved_output[cam_id].frames:
        for d in fd.detections:
            per_track_label[d.track_id].add(d.class_name)
    
    # Identify tracks that still have multiple labels (should ideally be only the ball track if the model is confused between ball and player)
    multi_label = [tid for tid, labels in per_track_label.items() if len(labels) > 1]

    # Print summary of label stability
    print(f"[{cam_id}] Tracks with a single stable label: {len(per_track_label) - len(multi_label)} / {len(per_track_label)}")
    if multi_label:
        print(f"  Tracks still showing multiple labels (should only be Ball): {multi_label}")

### Inspect Resolved Tracks

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], sort_resolved_output[INSPECT_CAMERA_ID])

---

## Tracking with DeepSORT

> Wojke, N., Bewley, A., & Paulus, D. (2017). Simple online and realtime tracking with a deep association metric. *arXiv:1703.07402*. [`deepsort`]
>
> Zhou, K., Yang, Y., Cavallaro, A., & Xiang, T. (2019). Omni-Scale Feature Learning for Person Re-Identification. *arXiv:1905.00953*. [`reid`]

The fine-tuned model gives us solid per-frame detections, but each frame is independent — there's no notion of identity across time. We pipe the merged detections through our DeepSORT implementation (IoU + Kalman filter + appearance ReID) so each player gets a stable `track_id` that persists frame-to-frame, surviving short occlusions up to `max_age` frames.

The tracker is intentionally class-agnostic: a track can absorb mislabeled observations across frames, which the next step will use to recover a single stable label per track.

### Parameters

In [ ]:
# Tuning notes for sports footage at 25 fps:
#   max_age:                 frames a track stays alive while unmatched before being
#                            killed. 30 ≈ 1.2 s of occlusion tolerance — too short for
#                            football where players cross for 2–3 s. We use 60 (≈ 2.4 s).
#   max_iou_distance:        IoU gating. Looser (0.8) tolerates more Kalman drift over
#                            longer gaps; too loose causes id swaps between nearby players.
#   max_appearance_distance: cosine distance gate for the OSNet appearance feature.
#   n_init:                  frames a tentative track must match before being CONFIRMED.
DEEPSORT_MAX_IOU_DISTANCE        = 0.8   # IoU gating. Looser (0.8) tolerates more Kalman drift over
DEEPSORT_MAX_APPEARANCE_DISTANCE = 0.2   # cosine distance threshold for appearance-based matching (lower = stricter, 0.2 is a common default)
DEEPSORT_MAX_AGE                 = 60    # frames a track stays alive while unmatched before being killed. 30 ≈ 1.2 s of occlusion tolerance — too short for football where players cross for 2–3 s. We use 60 (≈ 2.4 s).
DEEPSORT_N_INIT                  = 2     # consecutive matches required before a track is confirmed
DEEPSORT_FEATURE_BUDGET          = 100   # max appearance features stored per track

### Run DeepSORT Tracking on All Frames

In [ ]:
deepsort_tracking_output = {}
for cam_id in CAMERAS:
    # Apply DeepSORT tracking to the fine-tuned YOLO detections
    print(f"[{cam_id}] Running DeepSORT on {len(yolo_finetuned_detection_output[cam_id].frames)} frames...")
    start_time = time.time()
    deepsort_tracking_output[cam_id] = apply_deep_sort(
        yolo_finetuned_detection_output[cam_id],
        frames=frames_color[cam_id],
        max_iou_distance=DEEPSORT_MAX_IOU_DISTANCE,
        max_appearance_distance=DEEPSORT_MAX_APPEARANCE_DISTANCE,
        max_age=DEEPSORT_MAX_AGE,
        n_init=DEEPSORT_N_INIT,
        feature_budget=DEEPSORT_FEATURE_BUDGET,
    )
    elapsed = time.time() - start_time

    # Count total tracks and detections after tracking
    total_tracks = len({d.track_id for f in deepsort_tracking_output[cam_id].frames for d in f.detections})
    total_dets   = sum(len(f.detections) for f in deepsort_tracking_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {total_tracks} tracks across {total_dets} tracked detections")

### Inspect DeepSORT Tracks

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], deepsort_tracking_output[INSPECT_CAMERA_ID])

### Resolve Per-Track Labels

YOLO's class names already encode identity (e.g. `Red_11` = Red team #11), but the model occasionally flips a player's label between frames or returns two boxes with the same class in one frame. We use the track ids assigned above to vote for a stable label per track and resolve collisions where two tracks claim the same identity.

The track with the highest cumulative confidence keeps its top-choice label; any other track that wanted that label falls through to its next-best class. `Ball` is exempt — it's a category, not a unique identity.

In [ ]:
deepsort_resolved_output = {}
for cam_id in CAMERAS:
    # Resolve track labels by majority voting on the detections associated with each track
    deepsort_resolved_output[cam_id] = resolve_track_labels(deepsort_tracking_output[cam_id])
    
    # Analyze label stability per track (expecting most tracks to have a single stable label, ideally "Ball" or "Player")
    per_track_label = defaultdict(set)
    for fd in deepsort_resolved_output[cam_id].frames:
        for d in fd.detections:
            per_track_label[d.track_id].add(d.class_name)

    # Identify tracks that still have multiple labels (should ideally be only the ball track if the model is confused between ball and player)
    multi_label = [tid for tid, labels in per_track_label.items() if len(labels) > 1]
    print(f"[{cam_id}] Tracks with a single stable label: {len(per_track_label) - len(multi_label)} / {len(per_track_label)}")
    if multi_label:
        print(f"  Tracks still showing multiple labels (should only be Ball): {multi_label}")

### Inspect Resolved Labels

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], deepsort_resolved_output[INSPECT_CAMERA_ID])

---

# Evaluation

---

## Ground Truth

In [ ]:
# Download annotations if not already present (requires ANNOTATIONS_API_KEY in .env)
if not any(Path(ANNOTATIONS).glob("*.json")):
    load_dotenv()
    api_key = os.environ["ANNOTATIONS_API_KEY"]
    download_annotations("tracking-rybv7", "tracking_01", 1, api_key, ANNOTATIONS)
else:
    print(f"Annotations already present in {ANNOTATIONS} — skipping download.")

# Load annotations for each camera and print summary
ground_truth = {}
for cam_id in CAMERAS:
    ground_truth[cam_id] = load_annotations(camera_id=cam_id, version="tracking_01")
    n_ann = sum(len(f.detections) for f in ground_truth[cam_id].frames)
    print(f"[{cam_id}] Loaded {n_ann} annotations across {len(ground_truth[cam_id].frames)} frames")

---

## Detection

Detection metrics are computed at the bounding-box level, ignoring identity.

| Metric | Definition |
|--------|-----------|
| **TP** | Predicted boxes that match a ground-truth box (IoU ≥ 0.5) |
| **FP** | Predicted boxes with no matching ground-truth box |
| **FN** | Ground-truth boxes with no matching prediction |
| **Precision** | TP / (TP + FP) — how many detections are correct |
| **Recall** | TP / (TP + FN) — how many ground-truth boxes are found |
| **F1** | Harmonic mean of precision and recall |
| **Mean IoU** | Average overlap of matched pairs |

In [ ]:
# Evaluate detection performance of MOG2, pre-trained YOLO, and fine-tuned YOLO against ground truth annotations
detection_results = {}
for method_name, outputs in [
    ("MOG2",             mog2_detection_output),
    ("YOLO pre-trained", yolo_pretrained_detection_output),
    ("YOLO fine-tuned",  yolo_finetuned_detection_output),
]:
    for cam_id in CAMERAS:
        detection_results[(method_name, cam_id)] = evaluate_detection(
            ground_truth[cam_id], outputs[cam_id], frame_stride=FRAME_STRIDE,
        )

# Display detection results in a table
show_detection_table(detection_results)

---

## Tracking

> Luiten, J., et al. (2020). HOTA: A Higher Order Metric for Evaluating Multi-object Tracking. *IJCV*, 129(2), 548–578. [`hota`]

Tracking evaluation uses three complementary metric families:

**Detection-level** (same as above): measures per-frame bounding box quality regardless of identity.

**Identity (IDF1)**: measures how consistently each ground-truth identity is assigned the same track id over time.

| Metric | Definition |
|--------|-----------|
| **IDTP** | GT detections correctly matched to the same predicted id |
| **IDFP** | Predicted detections matched to the wrong GT identity |
| **IDFN** | GT detections not matched by any consistent id |
| **IDF1** | 2·IDTP / (2·IDTP + IDFP + IDFN) |

**HOTA**: decomposes tracking quality into detection accuracy (DetA) and association accuracy (AssA).

| Metric | Definition |
|--------|-----------|
| **HOTA** | √(DetA × AssA) — geometric mean averaged over IoU thresholds |
| **DetA** | Detection accuracy at each IoU threshold |
| **AssA** | Association accuracy at each IoU threshold |
| **LocA** | Localisation accuracy of matched pairs |

In [ ]:
# Evaluate tracking performance of SORT and DeepSORT against ground truth annotations
tracking_results = {}
for tracker_name, outputs in [
    ("SORT",     sort_resolved_output),
    ("DeepSORT", deepsort_resolved_output),
]:
    for cam_id in CAMERAS:
        tracking_results[(tracker_name, cam_id)] = evaluate_tracking(
            ground_truth[cam_id], outputs[cam_id], frame_stride=FRAME_STRIDE,
        )

### Identity Metrics

In [ ]:
show_identity_table(tracking_results)

### HOTA Metrics

In [ ]:
show_hota_table(tracking_results)